# NCKH — V5: VLM-Guided Native Negative Prompt Feedback for CountGD++

## Mục tiêu

V4 cho thấy Qwen3-VL có thể loại phần lớn hard-negative boxes nhưng vẫn có hai hạn chế:

1. phải verify rất nhiều box và từng có giới hạn số box được xem;
2. hard filtering vẫn có thể xóa nhầm positive boxes.

V5 thay đổi chiến lược: **không dùng Qwen để xóa từng box cuối cùng**. Thay vào đó, Qwen chỉ làm một việc rẻ hơn: **nhìn các box mà V4 đã reject và đặt tên cho confounder**. Tên confounder và một representative box sau đó được feed ngược vào **native negative prompt interface của CountGD++**.

```text
Image + Target
      ↓
V4 rejected-box evidence
      ↓
Qwen3-VL: "các box này thực ra là object gì?"
      ↓
confounder = ví dụ "cashew nuts"
      ↓
CountGD++ SECOND PASS
Positive text: "prawn crackers"
Negative text: "cashew nuts"
Negative visual exemplar: một crop cashew nuts
      ↓
CountGD++ tự contrast positive vs negative
      ↓
Final count
```

## Vì sao pipeline này đáng thử

CountGD++ chính thức hỗ trợ **negative text** và **negative visual exemplars**. Trong implementation chính thức, caption được ghép thành positive text + negative texts, sau đó detection được giữ khi positive logit lớn hơn negative logit. V5 tận dụng chính interface native này thay vì xóa box sau inference.

## V5 dùng lại kết quả V4

Để không phải chạy lại 15 giờ Qwen box verification:

- V5 đọc trực tiếp `nckh_v4_qwen3_detection_verification_export.zip`;
- chỉ dùng các box V4 đã đánh dấu `NO` làm evidence để **discover confounder**;
- Qwen3 chỉ gọi khoảng **1 lần / image-query group**, không gọi cho từng box;
- sau đó CountGD++ chạy second-pass trên toàn ảnh, nên **không còn cap 120 box**.

## Scientific note

V5 được thiết kế sau khi đã quan sát V4 trên cùng 111 hard negatives. Vì vậy V5 là **exploratory follow-up**, không phải untouched final test. Không dùng kết quả V5 trên 111 pairs để tune threshold/prompt nhiều vòng rồi gọi đó là final benchmark. Sau khi chốt pipeline cần một held-out negative set mới để xác nhận.


In [ ]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================

from pathlib import Path
import random

SEED = 42
random.seed(SEED)

# Luôn chạy SMOKE trước. Khi ổn thì đổi FULL.
RUN_MODE = "FULL"   # "SMOKE" | "FULL"

# CountGD++
COUNTGD_GITHUB = "https://github.com/niki-amini-naieni/CountGDPlusPlus.git"
COUNTGD_REPO = Path("/kaggle/working/CountGDPlusPlus")
COUNTGD_GDRIVE_ID = "1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs"
COUNTGD_THRESH = 0.23

# Qwen chỉ dùng để discover confounder, không verify từng box lại.
QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
DISCOVERY_REJECTED_CROPS = 6
PANEL_SIZE = 224
CROP_PADDING_FRAC = 0.30

# Primary V5 method được freeze TRƯỚC khi chạy test:
# negative text + one internal negative visual exemplar.
PRIMARY_METHOD = "NEG_TEXT_PLUS_EXEMPLAR"

# SMOKE sample size
SMOKE_HARD_OLD = 3
SMOKE_HARD_NEW = 3
SMOKE_POS = 4

SAVE_EVERY = 1
FAIL_FAST = False
RETRY_ERRORS = False

OUTPUT_DIR = Path("/kaggle/working/nckh_v5_native_negative_feedback")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

V4_EXTRACT_DIR = Path("/kaggle/working/v4_export_for_v5")
MANIFEST_CSV = OUTPUT_DIR / "v5_manifest.csv"
DISCOVERY_CSV = OUTPUT_DIR / "v5_confounder_discovery.csv"
FEEDBACK_CSV = OUTPUT_DIR / "v5_native_feedback_results.csv"
METRICS_CSV = OUTPUT_DIR / "v5_metrics.csv"
METRICS_ORIGIN_CSV = OUTPUT_DIR / "v5_metrics_by_origin.csv"
FAILURES_CSV = OUTPUT_DIR / "v5_failure_cases.csv"
SUMMARY_JSON = OUTPUT_DIR / "v5_summary.json"
BOOTSTRAP_JSON = OUTPUT_DIR / "v5_bootstrap.json"

print("RUN_MODE:", RUN_MODE)
print("PRIMARY_METHOD:", PRIMARY_METHOD)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision của Kaggle.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v5_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers>=4.57.0,<5",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# ============================================================
# CELL 3 — VERIFY CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V5 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS")


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


## Input cần Add vào Kaggle trước Cell 5

Chỉ cần 2 input chính:

1. **Dataset FSC-147** đã dùng ở V4, có `images_384_VarV2`.
2. **ZIP kết quả V4**: `nckh_v4_qwen3_detection_verification_export.zip`.

Không cần chạy lại V4 và không cần V1/V2 ZIP.


In [ ]:
# ============================================================
# CELL 5 — FIND INPUTS + EXTRACT V4 + BUILD MANIFEST
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import json

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(name):
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob(name) if p.is_file()])
    if not hits:
        return None
    return sorted(hits, key=lambda p: (0 if str(p).startswith("/kaggle/input") else 1, len(str(p))))[0]

def find_image_dir():
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits.extend([p for p in root.rglob("images_384_VarV2") if p.is_dir()])
    if not hits:
        return None
    return max(hits, key=lambda p: len(list(p.glob("*.jpg"))))

# ============================================================
# FIND V4 — hỗ trợ cả ZIP và Kaggle dataset đã giải nén
# ============================================================

V4_ZIP = find_file(
    "nckh_v4_qwen3_detection_verification_export.zip"
)

V4_SUMMARY_DIRECT = find_file(
    "v4_summary.json"
)

IMAGE_DIR = find_image_dir()

if IMAGE_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy images_384_VarV2"
    )


# ------------------------------------------------------------
# CASE 1: Có ZIP V4
# ------------------------------------------------------------
if V4_ZIP is not None:

    print("Found V4 ZIP:")
    print(V4_ZIP)

    V4_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(
        V4_ZIP,
        "r"
    ) as zf:
        zf.extractall(
            V4_EXTRACT_DIR
        )

    V4_ROOT = V4_EXTRACT_DIR


# ------------------------------------------------------------
# CASE 2: Kaggle đã giải nén V4 thành dataset
# ------------------------------------------------------------
elif V4_SUMMARY_DIRECT is not None:

    V4_ROOT = V4_SUMMARY_DIRECT.parent

    print("Found extracted V4 dataset:")
    print(V4_ROOT)


else:
    raise FileNotFoundError(
        "Không tìm thấy V4 ZIP hoặc v4_summary.json"
    )


# ------------------------------------------------------------
# Các file V4 cần dùng
# ------------------------------------------------------------
V4_PRED = (
    V4_ROOT
    / "v4_qwen3_predictions.csv"
)

V4_HARD_CAND = (
    V4_ROOT
    / "v4_hard_candidates.csv"
)

V4_HARD_METRICS = (
    V4_ROOT
    / "v4_hard_image_metrics.csv"
)

V4_POS_METRICS = (
    V4_ROOT
    / "v4_positive_image_metrics.csv"
)

V4_SUMMARY = (
    V4_ROOT
    / "v4_summary.json"
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------
required_v4_files = [
    V4_PRED,
    V4_HARD_CAND,
    V4_HARD_METRICS,
    V4_POS_METRICS,
    V4_SUMMARY,
]

for p in required_v4_files:

    if not p.exists():
        raise FileNotFoundError(
            f"Thiếu file V4: {p}"
        )

print("\n==============================")
print("V4 INPUT OK")
print("==============================")

print("V4_ROOT:")
print(V4_ROOT)

print("\nIMAGE_DIR:")
print(IMAGE_DIR)

print(
    "\nJPG count:",
    len(list(IMAGE_DIR.glob("*.jpg")))
)

for p in [V4_PRED, V4_HARD_CAND, V4_HARD_METRICS, V4_POS_METRICS, V4_SUMMARY]:
    if not p.exists():
        raise FileNotFoundError(p)

pred = pd.read_csv(V4_PRED)
hard_cand = pd.read_csv(V4_HARD_CAND)
hard_m = pd.read_csv(V4_HARD_METRICS)
pos_m = pd.read_csv(V4_POS_METRICS)

image_index = {p.name: str(p) for p in IMAGE_DIR.glob("*.jpg")}

# Hard manifest: 111 pairs.
hard = hard_m.merge(
    hard_cand[["pair_id","image_name","source_class","target","origin_batch"]],
    on=["pair_id","image_name","source_class","target","origin_batch"],
    how="left",
)
hard["group_id"] = hard["pair_id"].astype(str)
hard["pair_type"] = "HARD_NEGATIVE"
hard["gt_count"] = 0
hard["v4_filtered_count"] = hard["filtered_count"]
hard["image_path"] = hard["image_name"].map(image_index)

# Positive manifest: 96 images.
pos = pos_m.copy()
pos["group_id"] = "POS__" + pos["image_name"].astype(str)
pos["pair_type"] = "POSITIVE"
pos["source_class"] = pos["target"]
pos["origin_batch"] = "POSITIVE"
pos["v4_filtered_count"] = pos["filtered_count"]
pos["image_path"] = pos["image_name"].map(image_index)

common_cols = [
    "group_id","pair_type","image_name","image_path","source_class","target",
    "origin_batch","gt_count","baseline_count","v4_filtered_count"
]
manifest = pd.concat([hard[common_cols], pos[common_cols]], ignore_index=True)

if manifest["image_path"].isna().any():
    bad = manifest.loc[manifest["image_path"].isna(), "image_name"].tolist()
    raise FileNotFoundError(f"Thiếu ảnh: {bad[:20]}")

# SMOKE: luôn có old, new và positive.
if RUN_MODE == "SMOKE":
    old = manifest[(manifest["pair_type"]=="HARD_NEGATIVE") & (manifest["origin_batch"]=="V3_OLD_56")]
    new = manifest[(manifest["pair_type"]=="HARD_NEGATIVE") & (manifest["origin_batch"]=="V4_NEW_REVIEWED")]
    pp = manifest[manifest["pair_type"]=="POSITIVE"]
    manifest_run = pd.concat([
        old.sample(min(SMOKE_HARD_OLD, len(old)), random_state=SEED),
        new.sample(min(SMOKE_HARD_NEW, len(new)), random_state=SEED+1),
        pp.sample(min(SMOKE_POS, len(pp)), random_state=SEED+2),
    ], ignore_index=True)
else:
    manifest_run = manifest.copy()

manifest_run = manifest_run.sort_values(["pair_type","origin_batch","group_id"]).reset_index(drop=True)
manifest_run.to_csv(MANIFEST_CSV, index=False)

print("V4 ZIP:", V4_ZIP)
print("IMAGE_DIR:", IMAGE_DIR)
print("JPG count:", len(image_index))
print("\nFULL manifest:")
print(manifest["pair_type"].value_counts())
print("\nTHIS RUN:")
print(manifest_run["pair_type"].value_counts())
print(manifest_run["origin_batch"].value_counts())
display(manifest_run.head(12))


In [ ]:
# ============================================================
# CELL 6 — PREPARE REJECTED-BOX EVIDENCE FROM V4
# Không dùng source_class làm input cho Qwen.
# ============================================================

import pandas as pd
import numpy as np

pred_ok = pred[pred["status"].astype(str).eq("ok")].copy()
pred_ok["vlm_keep"] = pd.to_numeric(pred_ok["vlm_keep"], errors="coerce").fillna(1).astype(int)
pred_ok["score"] = pd.to_numeric(pred_ok["score"], errors="coerce").fillna(0.0)

# Chỉ evidence mà V4 verifier đã reject.
rejected = pred_ok[pred_ok["vlm_keep"].eq(0)].copy()
rejected = rejected[rejected["group_id"].astype(str).isin(set(manifest_run["group_id"].astype(str)))]

# Remap path về current Kaggle input.
rejected["image_path"] = rejected["image_name"].map(image_index)

print("Rejected evidence boxes in this run:", len(rejected))
print("Groups with >=1 rejected box:", rejected["group_id"].nunique(), "/", len(manifest_run))

coverage = manifest_run[["group_id","pair_type","target"]].copy()
counts = rejected.groupby("group_id").size().rename("rejected_boxes_v4")
coverage = coverage.merge(counts, on="group_id", how="left")
coverage["rejected_boxes_v4"] = coverage["rejected_boxes_v4"].fillna(0).astype(int)
display(coverage.sort_values("rejected_boxes_v4", ascending=False).head(15))


In [ ]:
# ============================================================
# CELL 7 — LOAD QWEN3-VL-4B FOR CONFOUNDER DISCOVERY
# ============================================================

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
qwen_model.eval()

print("Loaded:", QWEN_MODEL_ID)
print("Device:", next(qwen_model.parameters()).device)


In [ ]:
# ============================================================
# CELL 8 — DISCOVERY CONTACT SHEET + JSON PARSER
# ============================================================

from PIL import Image, ImageDraw, ImageOps
import json
import re
import math
import torch


def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    W,H = image.size
    x1,y1,x2,y2 = [float(v) for v in box]
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px = bw * padding_frac
    py = bh * padding_frac
    xx1 = int(max(0, math.floor(x1-px)))
    yy1 = int(max(0, math.floor(y1-py)))
    xx2 = int(min(W, math.ceil(x2+px)))
    yy2 = int(min(H, math.ceil(y2+py)))
    return image.crop((xx1,yy1,xx2,yy2)).convert("RGB")


def make_discovery_sheet(image, rows):
    n = len(rows)
    cols = 3
    rows_n = int(math.ceil(n / cols))
    label_h = 28
    sheet = Image.new("RGB", (cols*PANEL_SIZE, rows_n*(PANEL_SIZE+label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for local_id, row in enumerate(rows):
        box = json.loads(row["box_json"]) if isinstance(row["box_json"], str) else row["box_json"]
        crop = expanded_crop(image, box)
        crop = ImageOps.contain(crop, (PANEL_SIZE, PANEL_SIZE))
        x0 = (local_id % cols) * PANEL_SIZE
        y0 = (local_id // cols) * (PANEL_SIZE + label_h)
        panel = Image.new("RGB", (PANEL_SIZE, PANEL_SIZE), "white")
        panel.paste(crop, ((PANEL_SIZE-crop.width)//2, (PANEL_SIZE-crop.height)//2))
        sheet.paste(panel, (x0,y0))
        draw.rectangle([x0,y0,x0+PANEL_SIZE-1,y0+PANEL_SIZE+label_h-1], outline="black", width=2)
        draw.text((x0+6,y0+PANEL_SIZE+5), f"ID {local_id}", fill="black")
    return sheet


def extract_json_obj(text):
    s = str(text).strip()
    s = s.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, flags=re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None


def norm_label(s):
    s = re.sub(r"[^a-z0-9 ]+", " ", str(s).lower())
    toks = [t for t in s.split() if t not in {"a","an","the","object","objects","item","items"}]
    # crude singular normalization; chỉ là safety guard, không dùng để score model.
    out=[]
    for t in toks:
        if len(t)>4 and t.endswith("ies"):
            t=t[:-3]+"y"
        elif len(t)>3 and t.endswith("s") and not t.endswith("ss"):
            t=t[:-1]
        out.append(t)
    return " ".join(out).strip()


def labels_too_close(a,b):
    a=norm_label(a); b=norm_label(b)
    if not a or not b:
        return False
    if a==b or a in b or b in a:
        return True
    A=set(a.split()); B=set(b.split())
    if not A or not B:
        return False
    j=len(A&B)/len(A|B)
    return j>=0.75


@torch.inference_mode()
def qwen_discover_confounder(image, target, rows):
    sheet = make_discovery_sheet(image, rows)

    prompt = f"""
You are helping an object-counting system identify a NON-TARGET confounder.

Target category: {target}

The second image is a numbered contact sheet of detections that a previous verifier REJECTED as not being the target.
The first image is the full scene for context.

Decide whether the rejected crops are dominated by ONE clear non-target object category that could be confusing the counting model.

Rules:
- NEVER use a synonym, subtype, supertype, plural/singular variant, or obvious paraphrase of the target as a negative category.
- If the crops are actually the target, parts of the target, mixed unrelated things, background, or uncertain: return decision NONE.
- Use decision USE only when one visually coherent non-target category clearly explains the rejected detections.
- confounder must be a short common noun phrase.
- representative_id must point to the clearest panel showing the confounder.
- Do not use any hidden labels. Judge only the images and the target text.

Return exactly one JSON object and nothing else:
{{"decision":"USE","confounder":"cashew nuts","representative_id":2}}
or
{{"decision":"NONE","confounder":"","representative_id":-1}}
""".strip()

    messages=[{
        "role":"user",
        "content":[
            {"type":"image","image":image},
            {"type":"image","image":sheet},
            {"type":"text","text":prompt},
        ],
    }]

    text_prompt=qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas=None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs=list(video_inputs)
        video_metadatas=list(video_metadatas)

    inputs=qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    )
    inputs=inputs.to(next(qwen_model.parameters()).device)

    generated_ids=qwen_model.generate(**inputs, max_new_tokens=96, do_sample=False)
    trimmed=[out_ids[len(in_ids):] for in_ids,out_ids in zip(inputs.input_ids, generated_ids)]
    raw=qwen_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    obj=extract_json_obj(raw)
    return raw,obj,sheet

print("Discovery utilities ready.")


In [ ]:
# ============================================================
# CELL 9 — SANITY CHECK 3 GROUPS
# Không chạy tiếp FULL nếu output JSON không parse được.
# ============================================================

from PIL import Image

examples=[]
for _, mrow in manifest_run.iterrows():
    g=rejected[rejected["group_id"].astype(str).eq(str(mrow["group_id"]))].sort_values("score", ascending=False).head(DISCOVERY_REJECTED_CROPS)
    if len(g):
        examples.append((mrow,g))
    if len(examples)>=3:
        break

for mrow,g in examples:
    image=Image.open(mrow["image_path"]).convert("RGB")
    raw,obj,sheet=qwen_discover_confounder(image, str(mrow["target"]), g.to_dict("records"))
    print("\n========================================")
    print("group:", mrow["group_id"])
    print("pair_type:", mrow["pair_type"])
    print("target:", mrow["target"])
    print("RAW:", raw)
    print("PARSED:", obj)
    if obj and str(obj.get("decision","")).upper()=="USE":
        print("too_close_guard:", labels_too_close(obj.get("confounder",""), mrow["target"]))
    display(sheet)

print("\nNếu 3 case trên đều parse JSON ổn, mới chạy Cell 10.")


In [ ]:
# ============================================================
# CELL 10 — RUN CONFOUNDER DISCOVERY + RESUME
# ============================================================

import pandas as pd
import time
from pathlib import Path
from PIL import Image


def atomic_csv(df,path):
    tmp=Path(str(path)+".tmp")
    df.to_csv(tmp,index=False)
    tmp.replace(path)

existing=pd.read_csv(DISCOVERY_CSV) if DISCOVERY_CSV.exists() else pd.DataFrame()
records=existing.to_dict("records") if len(existing) else []

if len(existing):
    if RETRY_ERRORS:
        done=set(existing.loc[existing["status"].astype(str).eq("ok"),"group_id"].astype(str))
    else:
        done=set(existing["group_id"].astype(str))
else:
    done=set()

for idx,mrow in manifest_run.iterrows():
    gid=str(mrow["group_id"])
    if gid in done:
        continue

    g=(
        rejected[rejected["group_id"].astype(str).eq(gid)]
        .sort_values("score",ascending=False)
        .head(DISCOVERY_REJECTED_CROPS)
        .copy()
    )

    t0=time.time()
    raw=""; error=""; status="ok"
    decision="NONE"; confounder=""; rep_id=-1; rep_box=""

    if len(g)==0:
        status="no_rejected_evidence"
    else:
        try:
            image=Image.open(mrow["image_path"]).convert("RGB")
            raw,obj,_=qwen_discover_confounder(image,str(mrow["target"]),g.to_dict("records"))
            if obj is None:
                status="parse_error"
            else:
                decision=str(obj.get("decision","NONE")).upper().strip()
                confounder=str(obj.get("confounder","")).strip()
                try:
                    rep_id=int(obj.get("representative_id",-1))
                except Exception:
                    rep_id=-1

                # Conservative safety guards.
                if decision!="USE":
                    decision="NONE"; confounder=""; rep_id=-1
                elif not confounder:
                    decision="NONE"; rep_id=-1; status="empty_confounder"
                elif labels_too_close(confounder,mrow["target"]):
                    decision="NONE"; confounder=""; rep_id=-1; status="blocked_target_like_label"
                elif not (0<=rep_id<len(g)):
                    decision="NONE"; confounder=""; rep_id=-1; status="bad_representative_id"
                else:
                    rep_box=str(g.iloc[rep_id]["box_json"])
        except Exception as e:
            status="error"; error=repr(e)
            if FAIL_FAST:
                raise

    records.append({
        **{c:mrow[c] for c in ["group_id","pair_type","image_name","image_path","source_class","target","origin_batch","gt_count","baseline_count","v4_filtered_count"]},
        "evidence_boxes_n":int(len(g)),
        "decision":decision,
        "confounder":confounder,
        "representative_id":int(rep_id),
        "representative_box_json":rep_box,
        "raw_output":raw,
        "latency_sec":round(time.time()-t0,3),
        "status":status,
        "error":error,
    })
    done.add(gid)
    atomic_csv(pd.DataFrame(records),DISCOVERY_CSV)
    print(f"[{idx+1}/{len(manifest_run)}] {gid} | target={mrow['target']} | {decision} | conf={confounder!r} | {status}")

discovery=pd.DataFrame(records)
atomic_csv(discovery,DISCOVERY_CSV)

print("\nDiscovery decision:")
print(discovery["decision"].value_counts(dropna=False))
print("\nBy pair type:")
print(pd.crosstab(discovery["pair_type"],discovery["decision"]))
display(discovery[["group_id","pair_type","target","confounder","decision","status"]].head(20))


In [ ]:
# ============================================================
# CELL 11 — UNLOAD QWEN BEFORE COUNTGD++
# ============================================================

import gc
import torch

try:
    del qwen_model
    del qwen_processor
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()
print("Qwen unloaded; CUDA cache cleared.")


In [ ]:
# ============================================================
# CELL 12 — LOAD COUNTGD++ NATIVE NEGATIVE-PROMPT API
# Dùng functions từ official test_dataset.py.
# ============================================================

import os
import sys
import torch
from pathlib import Path

if str(COUNTGD_REPO) not in sys.path:
    sys.path.insert(0,str(COUNTGD_REPO))

script_path=COUNTGD_REPO/"test_dataset.py"
source=script_path.read_text(encoding="utf-8",errors="ignore")

# Chỉ exec phần định nghĩa function, dừng trước main script.
cut=source.find("logging.basicConfig")
if cut<0:
    raise RuntimeError("Không tìm được marker logging.basicConfig trong test_dataset.py")
source_defs=source[:cut]

ns={"__file__":str(script_path),"__name__":"countgd_v5_native_runtime"}
cwd0=os.getcwd()
os.chdir(str(COUNTGD_REPO))
try:
    exec(compile(source_defs,str(script_path),"exec"),ns)
finally:
    os.chdir(cwd0)

for k in ["build_arg_parser","build_model_and_transforms","count"]:
    if k not in ns:
        raise RuntimeError(f"Missing symbol: {k}")

args=ns["build_arg_parser"]().parse_args(["--pretrain_model_path",str(COUNTGD_CKPT)])
args.device="cuda:0"

_orig_load=torch.load
def _trusted_load(*a,**kw):
    kw.setdefault("weights_only",False)
    return _orig_load(*a,**kw)
torch.load=_trusted_load

cwd0=os.getcwd()
os.chdir(str(COUNTGD_REPO))
try:
    model,transform=ns["build_model_and_transforms"](args)
finally:
    os.chdir(cwd0)
    torch.load=_orig_load

device=torch.device("cuda:0")
model=model.to(device).eval()

# Set globals used by test_dataset._predict/count.
ns["model"]=model
ns["transform"]=transform
ns["device"]=device
ns["CONF_THRESH"]=float(COUNTGD_THRESH)

print("CountGD++ native negative-prompt API loaded.")
print("CONF_THRESH:",ns["CONF_THRESH"])


In [ ]:
# ============================================================
# CELL 13 — COUNTGD++ FEEDBACK WRAPPER
# Three frozen branches:
# 1) baseline: reused from V4
# 2) NEG_TEXT
# 3) NEG_TEXT_PLUS_EXEMPLAR (PRIMARY)
# ============================================================

from PIL import Image
import json
import numpy as np


def parse_box_json(x):
    if isinstance(x,(list,tuple)):
        return [float(v) for v in x]
    return [float(v) for v in json.loads(str(x))]


def countgd_feedback(image_path,target,confounder=None,rep_box_xyxy=None,use_exemplar=False):
    image=Image.open(image_path).convert("RGB")
    pos_prompts={"image":image,"points":[]}

    neg_texts=[]
    neg_prompts=[]

    if confounder:
        neg_texts=[str(confounder).strip()]

    if use_exemplar and confounder and rep_box_xyxy is not None:
        x1,y1,x2,y2=[float(v) for v in rep_box_xyxy]
        neg_prompts=[{
            "image":image,
            "points":[[x1,y1,2.0,x2,y2,3.0]],
        }]

    target_clean=str(target).replace(".","").strip()
    neg_texts=[str(x).replace(".","").strip() for x in neg_texts]

    # Use the official native _predict() path but skip count()'s heatmap
    # rendering, which would add unnecessary matplotlib overhead for hundreds
    # of second-pass runs.
    boxes_norm,scores=ns["_predict"](
        image,
        target_clean,
        pos_prompts,
        neg_texts,
        neg_prompts,
    )

    boxes_norm=np.asarray(boxes_norm)
    scores_arr=np.asarray(scores)
    if boxes_norm.size:
        x0,y0,bw,bh=ns["get_box_coords_from_boxes"](image,boxes_norm)
        boxes_arr=np.stack((x0,y0,bw,bh),axis=-1)
    else:
        boxes_arr=np.empty((0,4),dtype=float)

    return {
        "count":int(len(boxes_arr)),
        "boxes_xywh":boxes_arr.tolist(),
        "scores":scores_arr.tolist() if scores_arr.size else [],
    }

print("Feedback wrapper ready.")


In [ ]:
# ============================================================
# CELL 14 — SANITY CHECK ONE USE CASE
# Compare baseline vs NEG_TEXT vs NEG_TEXT+EXEMPLAR.
# ============================================================

use_rows=discovery[discovery["decision"].astype(str).eq("USE")].copy()
if len(use_rows)==0:
    print("Không có discovery USE trong SMOKE. Hãy kiểm tra Cell 9/10 trước khi chạy FULL.")
else:
    r=use_rows.iloc[0]
    box=parse_box_json(r["representative_box_json"])
    out_text=countgd_feedback(r["image_path"],r["target"],r["confounder"],None,False)
    out_both=countgd_feedback(r["image_path"],r["target"],r["confounder"],box,True)
    print("group:",r["group_id"])
    print("pair_type:",r["pair_type"])
    print("target:",r["target"])
    print("discovered confounder:",r["confounder"])
    print("V4 baseline count:",int(r["baseline_count"]))
    print("V4 filtered count:",int(r["v4_filtered_count"]))
    print("NEG_TEXT count:",out_text["count"])
    print("NEG_TEXT+EXEMPLAR count:",out_both["count"])


In [ ]:
# ============================================================
# CELL 15 — RUN NATIVE FEEDBACK ABLATIONS + RESUME
# If decision NONE: SAFE fallback = original baseline count.
# ============================================================

import pandas as pd
import time
import json

existing=pd.read_csv(FEEDBACK_CSV) if FEEDBACK_CSV.exists() else pd.DataFrame()
records=existing.to_dict("records") if len(existing) else []
done=set(existing["group_id"].astype(str)) if len(existing) else set()

for idx,r in discovery.iterrows():
    gid=str(r["group_id"])
    if gid in done:
        continue

    baseline=int(r["baseline_count"])
    text_count=baseline
    both_count=baseline
    text_boxes=[]; both_boxes=[]; text_scores=[]; both_scores=[]
    status="skip_no_confounder"; error=""; latency_text=0.0; latency_both=0.0

    if str(r["decision"]).upper()=="USE" and str(r["confounder"]).strip():
        try:
            rep_box=parse_box_json(r["representative_box_json"])

            t0=time.time()
            a=countgd_feedback(r["image_path"],r["target"],r["confounder"],None,False)
            latency_text=time.time()-t0
            text_count=a["count"]; text_boxes=a["boxes_xywh"]; text_scores=a["scores"]

            t0=time.time()
            b=countgd_feedback(r["image_path"],r["target"],r["confounder"],rep_box,True)
            latency_both=time.time()-t0
            both_count=b["count"]; both_boxes=b["boxes_xywh"]; both_scores=b["scores"]
            status="ok"
        except Exception as e:
            status="error"; error=repr(e)
            # fail-open: giữ baseline, không force 0
            text_count=baseline; both_count=baseline
            if FAIL_FAST:
                raise

    rec={
        **{c:r[c] for c in ["group_id","pair_type","image_name","image_path","source_class","target","origin_batch","gt_count","baseline_count","v4_filtered_count","decision","confounder","representative_box_json"]},
        "neg_text_count":int(text_count),
        "neg_text_exemplar_count":int(both_count),
        "neg_text_boxes_xywh_json":json.dumps(text_boxes),
        "neg_text_scores_json":json.dumps(text_scores),
        "neg_text_exemplar_boxes_xywh_json":json.dumps(both_boxes),
        "neg_text_exemplar_scores_json":json.dumps(both_scores),
        "latency_neg_text_sec":round(latency_text,3),
        "latency_neg_text_exemplar_sec":round(latency_both,3),
        "status":status,
        "error":error,
    }
    records.append(rec)
    done.add(gid)
    atomic_csv(pd.DataFrame(records),FEEDBACK_CSV)
    print(f"[{idx+1}/{len(discovery)}] {gid} | {status} | base={baseline} | text={text_count} | both={both_count} | conf={r['confounder']!r}")

feedback=pd.DataFrame(records)
atomic_csv(feedback,FEEDBACK_CSV)
print("Saved:",FEEDBACK_CSV)


In [ ]:
# ============================================================
# CELL 16 — METRICS: BASELINE vs V4 vs V5
# PRIMARY = NEG_TEXT_PLUS_EXEMPLAR, pre-specified before test.
# ============================================================

import pandas as pd
import numpy as np

feedback=pd.read_csv(FEEDBACK_CSV)
for c in ["gt_count","baseline_count","v4_filtered_count","neg_text_count","neg_text_exemplar_count"]:
    feedback[c]=pd.to_numeric(feedback[c],errors="coerce")

# Safe hybrid: native feedback when Qwen discovered a confounder,
# otherwise original baseline (NOT V4 hard filtering) to protect positives.
feedback["v5_primary_count"]=feedback["neg_text_exemplar_count"]

# Exploratory alternative only: if no confounder, fall back to V4 filtered result.
feedback["v5_hybrid_v4_fallback_count"]=np.where(
    feedback["decision"].astype(str).eq("USE"),
    feedback["neg_text_exemplar_count"],
    feedback["v4_filtered_count"],
)

methods={
    "COUNTGD_BASELINE":"baseline_count",
    "V4_BOX_FILTER":"v4_filtered_count",
    "V5_NEG_TEXT":"neg_text_count",
    "V5_NEG_TEXT_PLUS_EXEMPLAR":"neg_text_exemplar_count",
    "V5_HYBRID_V4_FALLBACK_EXPLORATORY":"v5_hybrid_v4_fallback_count",
}

rows=[]
for pair_type,g in feedback.groupby("pair_type"):
    for method,col in methods.items():
        y=g["gt_count"].to_numpy(float)
        p=g[col].to_numpy(float)
        row={
            "pair_type":pair_type,
            "method":method,
            "n":len(g),
            "mae":float(np.mean(np.abs(p-y))),
            "rmse":float(np.sqrt(np.mean((p-y)**2))),
            "mean_gt":float(np.mean(y)),
            "mean_pred":float(np.mean(p)),
        }
        if pair_type=="HARD_NEGATIVE":
            row["false_activation_rate"] = float(np.mean(p>0))
            row["zero_rate"] = float(np.mean(p==0))
        else:
            row["positive_survival_rate"] = float(np.mean(p>0))
        rows.append(row)

metrics=pd.DataFrame(rows)
metrics.to_csv(METRICS_CSV,index=False)
print("=== MAIN METRICS ===")
display(metrics)

# OLD_56 vs NEW_55 for hard negatives.
origin_rows=[]
hard=feedback[feedback["pair_type"].eq("HARD_NEGATIVE")].copy()
for origin,g in hard.groupby("origin_batch"):
    for method,col in methods.items():
        p=g[col].to_numpy(float)
        origin_rows.append({
            "origin_batch":origin,
            "method":method,
            "n":len(g),
            "mae":float(np.mean(np.abs(p))),
            "fpr":float(np.mean(p>0)),
            "mean_pred":float(np.mean(p)),
            "zero_rate":float(np.mean(p==0)),
        })
metrics_origin=pd.DataFrame(origin_rows)
metrics_origin.to_csv(METRICS_ORIGIN_CSV,index=False)
print("\n=== HARD NEGATIVE BY ORIGIN ===")
display(metrics_origin)

print("\n=== CONFOUNDER DISCOVERY COVERAGE ===")
print(pd.crosstab(feedback["pair_type"],feedback["decision"],normalize="index"))


In [ ]:
# ============================================================
# CELL 17 — PAIRED BOOTSTRAP + FAILURE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
import json

rng=np.random.default_rng(SEED)

def bootstrap_diff(a,b,n_boot=5000):
    a=np.asarray(a,float); b=np.asarray(b,float)
    assert len(a)==len(b)
    n=len(a)
    point=float(np.mean(a-b))
    vals=np.empty(n_boot,float)
    for i in range(n_boot):
        idx=rng.integers(0,n,size=n)
        vals[i]=np.mean(a[idx]-b[idx])
    lo,hi=np.quantile(vals,[0.025,0.975])
    return {"n":int(n),"point_estimate_A_minus_B":point,"ci95_low":float(lo),"ci95_high":float(hi)}

hard=feedback[feedback["pair_type"].eq("HARD_NEGATIVE")].copy()
pos=feedback[feedback["pair_type"].eq("POSITIVE")].copy()

boot={
    "interpretation":"Negative means V5 primary has lower absolute error than comparator.",
    "hard_V5_minus_baseline":bootstrap_diff(
        np.abs(hard["v5_primary_count"]-hard["gt_count"]),
        np.abs(hard["baseline_count"]-hard["gt_count"]),
    ),
    "hard_V5_minus_V4":bootstrap_diff(
        np.abs(hard["v5_primary_count"]-hard["gt_count"]),
        np.abs(hard["v4_filtered_count"]-hard["gt_count"]),
    ),
    "positive_V5_minus_baseline":bootstrap_diff(
        np.abs(pos["v5_primary_count"]-pos["gt_count"]),
        np.abs(pos["baseline_count"]-pos["gt_count"]),
    ),
    "positive_V5_minus_V4":bootstrap_diff(
        np.abs(pos["v5_primary_count"]-pos["gt_count"]),
        np.abs(pos["v4_filtered_count"]-pos["gt_count"]),
    ),
}
with open(BOOTSTRAP_JSON,"w",encoding="utf-8") as f:
    json.dump(boot,f,indent=2)
print(json.dumps(boot,indent=2))

# Failure table: biggest V5 degradation/improvement relative to V4 and baseline.
feedback["baseline_abs_error"]=(feedback["baseline_count"]-feedback["gt_count"]).abs()
feedback["v4_abs_error"]=(feedback["v4_filtered_count"]-feedback["gt_count"]).abs()
feedback["v5_abs_error"]=(feedback["v5_primary_count"]-feedback["gt_count"]).abs()
feedback["v5_vs_baseline_improvement"]=feedback["baseline_abs_error"]-feedback["v5_abs_error"]
feedback["v5_vs_v4_improvement"]=feedback["v4_abs_error"]-feedback["v5_abs_error"]

failure_cols=[
    "group_id","pair_type","image_name","source_class","target","origin_batch","gt_count",
    "decision","confounder","baseline_count","v4_filtered_count","neg_text_count",
    "neg_text_exemplar_count","baseline_abs_error","v4_abs_error","v5_abs_error",
    "v5_vs_baseline_improvement","v5_vs_v4_improvement","status"
]
failures=pd.concat([
    feedback.sort_values("v5_vs_v4_improvement").head(30),
    feedback.sort_values("v5_vs_v4_improvement",ascending=False).head(30),
],ignore_index=True).drop_duplicates("group_id")
failures[failure_cols].to_csv(FAILURES_CSV,index=False)

print("\n=== MOST DEGRADED VS V4 ===")
display(feedback.sort_values("v5_vs_v4_improvement")[failure_cols].head(15))
print("\n=== MOST IMPROVED VS V4 ===")
display(feedback.sort_values("v5_vs_v4_improvement",ascending=False)[failure_cols].head(15))


In [ ]:
# ============================================================
# CELL 18 — SUMMARY + EXPORT ZIP
# ============================================================

import json
import shutil
from pathlib import Path
from IPython.display import FileLink, display

feedback=pd.read_csv(FEEDBACK_CSV)
metrics=pd.read_csv(METRICS_CSV)
metrics_origin=pd.read_csv(METRICS_ORIGIN_CSV)

summary={
    "phase":"V5_NATIVE_NEGATIVE_PROMPT_FEEDBACK",
    "run_mode":RUN_MODE,
    "primary_method":PRIMARY_METHOD,
    "qwen_model":QWEN_MODEL_ID,
    "countgd_threshold":COUNTGD_THRESH,
    "design":{
        "confounder_source":"V4 rejected detection crops only; source_class is NOT provided to Qwen",
        "negative_text":"Qwen-discovered confounder",
        "negative_visual_exemplar":"one representative rejected V4 box chosen by Qwen",
        "fallback_when_no_confounder":"original CountGD baseline (fail-open)",
    },
    "n_groups":int(len(feedback)),
    "pair_type_counts":feedback["pair_type"].value_counts().to_dict(),
    "discovery_USE_rate_by_pair_type":{
        k:float(v) for k,v in (
            feedback.assign(use=feedback["decision"].astype(str).eq("USE").astype(int))
            .groupby("pair_type")["use"].mean().to_dict().items()
        )
    },
    "metrics":metrics.to_dict("records"),
    "metrics_by_origin":metrics_origin.to_dict("records"),
    "bootstrap":boot,
    "scientific_notes":[
        "V5 is exploratory because it was designed after observing V4 on the same 111 hard-negative benchmark.",
        "The primary method NEG_TEXT_PLUS_EXEMPLAR was frozen before running the evaluation.",
        "source_class is retained only for diagnostics and is not passed to Qwen confounder discovery or CountGD++ as a negative label.",
        "When discovery returns NONE or fails, V5 fails open to the original CountGD baseline to protect positives.",
        "A fresh held-out negative benchmark is required before calling V5 a final confirmatory result.",
    ],
}

with open(SUMMARY_JSON,"w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)

archive=shutil.make_archive(
    "/kaggle/working/nckh_v5_native_negative_feedback_export",
    "zip",
    root_dir=str(OUTPUT_DIR),
)

print(json.dumps(summary,indent=2,ensure_ascii=False)[:15000])
print("\nCreated:",archive)
print("Size MB:",round(Path(archive).stat().st_size/1024**2,2))
display(FileLink(archive))


## Cách đọc V5

Ưu tiên 4 câu hỏi:

1. Trên `HARD_NEGATIVE`, `V5_NEG_TEXT_PLUS_EXEMPLAR` có làm **FPR/MAE thấp hơn V4_BOX_FILTER** hay không?
2. Trên `POSITIVE`, V5 có kéo MAE về gần `COUNTGD_BASELINE` hơn V4 không?
3. `NEW_55` có còn cải thiện tương tự `OLD_56` không?
4. `discovery USE rate` trên positive có quá cao không? Nếu quá cao, Qwen vẫn đang tìm confounder cả khi các rejected boxes thật ra là target.

### Kết quả mong muốn

V5 đáng giữ nếu đồng thời:

- hard-negative FPR/MAE không tệ hơn V4 nhiều, tốt nhất là thấp hơn;
- positive MAE tốt hơn V4 rõ rệt;
- positive survival cao;
- NEW_55 vẫn giữ tín hiệu tốt;
- số lỗi discovery target-like bị safety guard chặn ở mức hợp lý.

### Nếu V5 native feedback chưa đủ

Bước sau mới thêm selective final verification chỉ trên **residual boxes sau second pass**, thay vì verify toàn bộ box như V4. Như vậy số Qwen calls sẽ giảm mạnh.
